# Real-data analysis: fates and marker tables

Reproduces the paper's DE tables: for each trajectory cluster, genes ranked by a Wilcoxon
rank-sum test of that cluster's cells against all other cells, at two time points --
**final** (the trajectory endpoint mapped back to gene space through the PCA loadings) and
**initial** (the same cells' observed expression at the first time point). The finished
rankings ship with the repository; the top 20 per cluster are printed below.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

_here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
# this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
for _d in (_here, os.path.abspath(os.path.join(_here, os.pardir, os.pardir, "tools"))):
    if _d not in sys.path:
        sys.path.insert(0, _d)
import _analysis_common as A
import _repo as P
import _fate_sets as FSETS

REPO    = A.ROOT
DATASET = globals().get("DATASET", "embryoid")    # "embryoid" | "statefate" -- run once each
DIM     = globals().get("DIM", 20)
SEED    = globals().get("SEED", {"embryoid": "5", "statefate": "2"}[DATASET])  # seed of record
SHIPPED = globals().get("SHIPPED", 1)   # 1 = read the shipped paper results; 0 = your new_results/ run
SAVE    = globals().get("SAVE", 0)      # used only by the optional DE recompute at the end

TOPN    = 20
SPEC    = {"embryoid": FSETS.EMBRYOID, "statefate": FSETS.STATEFATE}[DATASET]
RESULTS = A.results_dir(DATASET, DIM, shipped=SHIPPED)
SEEDSEL = os.path.join(os.path.dirname(RESULTS), "seed_selection")
TABLES  = (P.shipped if SHIPPED else P.results)("figures", "realdata", "tables")
print(f"[{DATASET} d={DIM} seed{SEED}] top {TOPN} per cluster | tables <- "
      f"{os.path.relpath(TABLES, REPO)}")

## The cluster ids and the settled fate calls
K-means ids are arbitrary, so every table is printed under the cluster ids the paper uses,
recovered from the two shipped label files. `_fate_sets.py` holds the settled fate call of each
cluster and the genes the text quotes for it -- a `*` in the tables marks those genes.

In [ ]:
def cluster_id_map():
    """{raw K-means id -> the cluster id the paper uses}, recovered from the two label files."""
    lab_pub = np.load(P.find("figures", "realdata", "trajs_comparisons",
                             f"labels_published_{DATASET}{DIM}_seed{SEED}.npy"))
    lab_raw = np.load(os.path.join(SEEDSEL, f"labels_seed{SEED}_all.npy"))
    K = int(lab_pub.max()) + 1
    m = {r: int(np.bincount(lab_pub[lab_raw == r], minlength=K).argmax()) for r in range(K)}
    assert len(set(m.values())) == K, m
    return m


def fate_info():
    """{cluster id -> (fate name, sub-name, is_fate, {when: [settled genes]})}."""
    out = {}
    for f in SPEC["fates"]:
        for pub in f["pub"]:
            out[int(pub)] = dict(fate=f["fate"], sub=f["sub"].get(pub, ""), is_fate=f["is_fate"],
                                 genes={"final": [g.upper() for g in f.get("final", {}).get(pub, [])],
                                        "initial": [g.upper() for g in
                                                    f.get("initial", {}).get(pub, [])]})
    return out


def load_de_table(when):
    """The shipped full Wilcoxon ranking for one time point ('final' | 'initial')."""
    df = pd.read_csv(os.path.join(TABLES, f"de_{DATASET}{DIM}_seed{SEED}_{when}_full.csv"))
    df["rank"] = df.groupby("group")["scores"].rank(ascending=False, method="min").astype(int)
    return df


MP, FI = cluster_id_map(), fate_info()
INV = {v: k for k, v in MP.items()}
for c in sorted(INV):
    info = FI.get(c, {})
    name = info.get("sub") or info.get("fate", "?")
    print(f"  cluster {c}: {name}" + ("" if info.get("is_fate", True) else "   (not a fate)"))

## At a glance
The first five genes of each cluster at each time point (`*` = named for that cluster in
`_fate_sets.py`); the full top-20 tables follow.

In [ ]:
DFS = {w: load_de_table(w) for w in ("final", "initial")}
print(f"  {'cluster':8s}{'fate':40s}{'top genes, final':44s}top genes, initial")
for c in sorted(INV):
    info = FI.get(c, {})
    nm = (info.get("sub") or info.get("fate", "?")) + ("" if info.get("is_fate", True) else " (not a fate)")
    cells = []
    for w in ("final", "initial"):
        sub = DFS[w][DFS[w].group == INV[c]].sort_values("rank").head(5)
        cells.append(", ".join(str(g) + ("*" if str(g).upper() in info.get("genes", {}).get(w, [])
                                         else "") for g in sub["names"]))
    print(f"  {c:<8d}{nm:40s}{cells[0]:44s}{cells[1]}")

## The top-20 tables (the Supplement's layout)
Blocks of clusters, each `gene | Z`; scores are the Wilcoxon Z statistics. The clusters were
built from the trajectories these genes are read off, so read scores as effect sizes describing
the clustering, not as inference.

In [ ]:
def print_tables(when, topn=TOPN, ncol=None):
    pubs = sorted(INV)
    ncol = ncol or (3 if len(pubs) % 3 == 0 else 2)
    df = DFS[when]
    print(f"\n[{DATASET} seed{SEED}] {when} time point "
          f"({'trajectory endpoint' if when == 'final' else 'first observed day'}), top {topn}:")
    for start in range(0, len(pubs), ncol):
        cols = pubs[start:start + ncol]
        print("  " + "".join(f"{'cluster ' + str(c) + ' gene':22s}{'Z':>8s}    " for c in cols))
        subs = [df[df.group == INV[c]].sort_values("rank").head(topn) for c in cols]
        for i in range(topn):
            row = ""
            for c, sub in zip(cols, subs):
                if i < len(sub):
                    g = str(sub.iloc[i]["names"])
                    g += "*" if g.upper() in FI.get(c, {}).get("genes", {}).get(when, []) else ""
                    row += f"{g:22s}{sub.iloc[i]['scores']:8.2f}    "
                else:
                    row += " " * 34
            print("  " + row)
        print()


print_tables("final")
print_tables("initial")